# Generate Submission From Trained Adapter

This notebook treats the adapter as fully trained and only runs inference to create `submission.csv`.

In [ ]:
!pip -q install -U "transformers>=4.57.6" "peft>=0.12.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.1" "datasets>=2.20.0"

In [ ]:
from dataclasses import dataclass
from pathlib import Path
import json
import math
import zipfile

import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

pd.set_option('display.max_colwidth', 180)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [ ]:
_KAGGLE_MODEL_PATH = '/kaggle/input/models/google/gemma-2/pytorch/gemma-2-9b-it/1'

@dataclass
class Config:
    base_model = _KAGGLE_MODEL_PATH if Path(_KAGGLE_MODEL_PATH).exists() else 'google/gemma-2-9b-it'
    adapter_zip = './adapter.zip'
    adapter_path = './adapter'
    test_csv = './test.csv'
    submission_path = './submission.csv'

    max_seq_len = 2048
    batch_size = 16
    tta = False

cfg = Config()

If you are running in Colab and your files are on Drive, update the paths in `cfg` before continuing.

In [ ]:
adapter_path = Path(cfg.adapter_path)
adapter_zip = Path(cfg.adapter_zip)

if not (adapter_path / 'adapter_config.json').exists() or not (adapter_path / 'adapter_model.safetensors').exists():
    if not adapter_zip.exists():
        raise FileNotFoundError(f'Could not find {adapter_path} or {adapter_zip}')

    with zipfile.ZipFile(adapter_zip) as zf:
        zf.extractall(adapter_path.parent)

    if not adapter_path.exists():
        extracted_dirs = [p for p in adapter_path.parent.iterdir() if p.is_dir() and (p / 'adapter_config.json').exists()]
        if extracted_dirs:
            adapter_path = extracted_dirs[0]

print('Using adapter path:', adapter_path.resolve())

In [ ]:
def flatten_text(x):
    if x is None:
        return ''
    if isinstance(x, float) and math.isnan(x):
        return ''
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return ''
        try:
            x = json.loads(s)
        except Exception:
            return s

    if isinstance(x, dict):
        x = list(x.values())
    if isinstance(x, (list, tuple)):
        parts = []
        for v in x:
            if v is None:
                continue
            if isinstance(v, float) and math.isnan(v):
                continue
            parts.append(str(v).strip())
        return ' '.join(p for p in parts if p)
    return str(x).strip()

def tokenize(tokenizer, prompt, response_a, response_b, max_length=cfg.max_seq_len):
    prompt = flatten_text(prompt)
    response_a = flatten_text(response_a)
    response_b = flatten_text(response_b)

    chunk = max_length // 3
    p_ids = tokenizer('<prompt>: ' + prompt, max_length=chunk, truncation=True, padding=False).input_ids
    ra_ids = tokenizer('<response_a>: ' + response_a, max_length=chunk, truncation=True, padding=False).input_ids
    rb_ids = tokenizer('<response_b>: ' + response_b, max_length=chunk, truncation=True, padding=False).input_ids

    input_ids = p_ids + ra_ids + rb_ids
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids)}

In [ ]:
test_df = pd.read_csv(cfg.test_csv)

for col in ['prompt', 'response_a', 'response_b']:
    test_df[col] = test_df[col].apply(flatten_text)

test_df.head()

In [ ]:
import json
from transformers import AutoConfig

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Patch missing model_type in Kaggle local config.json
extra_kwargs = {}
if Path(cfg.base_model).exists():
    config_files = list(Path(cfg.base_model).rglob('config.json'))
    if not config_files:
        raise FileNotFoundError(f'No config.json found under {cfg.base_model}')
    config_path = config_files[0]
    model_dir = str(config_path.parent)
    config_data = json.loads(config_path.read_text())
    config_data.setdefault('model_type', 'gemma2')
    config_data['num_labels'] = 3
    extra_kwargs['config'] = AutoConfig.for_model(config_data.pop('model_type'), **config_data)
else:
    model_dir = cfg.base_model

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_dir,
    num_labels=3,
    quantization_config=bnb_config,
    device_map='auto',
    **extra_kwargs,
)

base_model.config.pad_token_id = tokenizer.pad_token_id
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
def make_test_data(src_df, swap=False):
    ids, input_ids, masks = [], [], []
    for _, row in src_df.iterrows():
        ra = row['response_b'] if swap else row['response_a']
        rb = row['response_a'] if swap else row['response_b']
        enc = tokenize(tokenizer, row['prompt'], ra, rb)
        ids.append(row['id'])
        input_ids.append(enc['input_ids'])
        masks.append(enc['attention_mask'])
    df = pd.DataFrame({'id': ids, 'input_ids': input_ids, 'attention_mask': masks})
    df['length'] = df['input_ids'].apply(len)
    return df

_first_device = next(model.parameters()).device

@torch.no_grad()
@torch.cuda.amp.autocast()
def inference(df):
    df = df.sort_values('length', ascending=False).reset_index(drop=True)
    a_win, b_win, tie = [], [], []
    for i in range(0, len(df), cfg.batch_size):
        batch = df.iloc[i:i + cfg.batch_size]
        inputs = tokenizer.pad(
            {'input_ids': batch['input_ids'].tolist(), 'attention_mask': batch['attention_mask'].tolist()},
            padding=True,
            return_tensors='pt',
        ).to(_first_device)
        proba = model(**inputs).logits.softmax(-1).cpu()
        a_win.extend(proba[:, 0].tolist())
        b_win.extend(proba[:, 1].tolist())
        tie.extend(proba[:, 2].tolist())
    df['winner_model_a'] = a_win
    df['winner_model_b'] = b_win
    df['winner_tie'] = tie
    return df

In [ ]:
data = make_test_data(test_df)
result_df = inference(data)
proba = result_df[['winner_model_a', 'winner_model_b', 'winner_tie']].values

if cfg.tta:
    aug_data = make_test_data(test_df, swap=True)
    tta_result_df = inference(aug_data)
    tta_proba = tta_result_df[['winner_model_b', 'winner_model_a', 'winner_tie']].values
    proba = (proba + tta_proba) / 2

sub = pd.DataFrame({
    'id': result_df['id'],
    'winner_model_a': proba[:, 0],
    'winner_model_b': proba[:, 1],
    'winner_tie': proba[:, 2],
})

sub.to_csv(cfg.submission_path, index=False)
print('Saved:', Path(cfg.submission_path).resolve())
sub.head()